# 05b — Анализ CatBoost-модели скорости (travel_time_opt.cbm)

Загружаем обученную модель из notebook 05 и делаем полный анализ для диссертации:
- Метрики и сравнительная таблица
- Feature importance + SHAP
- Кривые скорость-уклон (сравнение с Tobler)
- Эффект тропы / дороги / воды
- Partial Dependence Plots
- Анализ ошибок по типам местности

In [ ]:
import pickle, warnings
from pathlib import Path
from math import radians, cos, degrees, atan2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings('ignore')

CACHE_DIR  = Path('cache')
MODELS_DIR = Path('models')

def tobler_kmh(slope_deg):
    s = np.asarray(slope_deg, dtype=float)
    return 6 * np.exp(-3.5 * np.abs(np.tan(np.radians(s)) + 0.05))

print('OK')


## 1. Загрузка данных и модели

In [ ]:
# Данные (из notebook 05)
df_raw = pickle.load(open(CACHE_DIR / 'hikr_model_df.pkl', 'rb'))
osm_labels = pickle.load(open(CACHE_DIR / 'hikr_osm_labels.pkl', 'rb'))

alps_mask = (
    (df_raw['lat_mid'] >= 43) & (df_raw['lat_mid'] <= 49) &
    (df_raw['lon_mid'] >= 5)  & (df_raw['lon_mid'] <= 20)
)
df = df_raw[alps_mask].copy().reset_index(drop=True)

df['dist_trail_m']   = osm_labels['dist_trail_m']
df['dist_road_m']    = osm_labels['dist_road_m']
df['dist_water_m']   = osm_labels['dist_water_m']
df['on_trail']       = (df['dist_trail_m'] < 50).astype(int)
df['on_road']        = (df['dist_road_m']  < 30).astype(int)
df['near_water']     = (df['dist_water_m'] < 100).astype(int)
df['off_trail']      = ((df['on_trail'] == 0) & (df['on_road'] == 0)).astype(int)
df['log_dist_trail'] = np.log1p(df['dist_trail_m'].clip(0, 5000))
df['log_dist_road']  = np.log1p(df['dist_road_m'].clip(0, 5000))

slope = df['slope_deg'].values
road  = df['on_road'].values.astype(bool)
trail = df['on_trail'].values.astype(bool)
water = df['near_water'].values.astype(bool)
base  = np.where(road, 1,
        np.where(trail & (slope < 15), 2,
        np.where(trail | (slope < 25), 3, 4)))
df['custom_difficulty'] = np.where(water & ~road, np.minimum(base + 1, 5), base).astype(int)
df['signed_slope_deg'] = np.degrees(np.arctan2(
    df['elev_diff_m'].values,
    np.maximum(df['dist_m'].values, 0.1)
))

FEATURES = [
    'signed_slope_deg', 'elev_diff_m',
    'on_trail', 'on_road', 'off_trail', 'near_water',
    'log_dist_trail', 'log_dist_road',
    'custom_difficulty',
]
CAT_F  = ['on_trail', 'on_road', 'off_trail', 'near_water', 'custom_difficulty']
TARGET = 'speed_kmh'

X = df[FEATURES].copy()
y = df[TARGET].copy()
groups = df['track_id'].values

# Canonical split (shared: NB-05, NB-06) - загружается, не пересоздаётся
_sp = pickle.load(open(CACHE_DIR / 'canonical_split.pkl', 'rb'))
tr_ids, val_ids, te_ids = _sp['tr_ids'], _sp['val_ids'], _sp['te_ids']
tr_mask = df['track_id'].isin(tr_ids).values
te_mask = df['track_id'].isin(te_ids).values
tr_idx  = np.where(tr_mask)[0]
te_idx  = np.where(te_mask)[0]
X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

# Загрузка модели
model_opt = CatBoostRegressor()
model_opt.load_model(str(MODELS_DIR / 'travel_time_opt.cbm'))

model_base = CatBoostRegressor()
model_base.load_model(str(MODELS_DIR / 'travel_time_custom.cbm'))

yp_opt  = model_opt.predict(X_te)
yp_base = model_base.predict(X_te)

print(f'Test сегментов: {len(y_te):,}')
print(f'Оптимизированная (opt):  MAE={mean_absolute_error(y_te,yp_opt):.3f}  R^2={r2_score(y_te,yp_opt):.3f}')
print(f'Базовая (base):          MAE={mean_absolute_error(y_te,yp_base):.3f}  R^2={r2_score(y_te,yp_base):.3f}')

In [ ]:
# === LSTM (NB-06) - загрузка модели + per-segment инференс ===
import time as _time
import torch, torch.nn as nn, torch.nn.functional as F

LSTM_POINT_FEAT = [
    'signed_slope_deg', 'elev_diff_m', 'dist_m',
    'on_trail', 'on_road', 'off_trail', 'near_water',
    'log_dist_trail', 'log_dist_road', 'custom_difficulty',
]
LSTM_ATTR_FEAT = [
    'elemax', 'elemin', 'D_plus', 'D_minus', 'disttotal',
    'vhat_n30', 'vhat_n20', 'vhat_n10', 'vhat_0', 'vhat_p10', 'vhat_p20', 'vhat_p30',
]
_SPLIT = 0.3
_MIN_S = 20

class HikingTTEGIS(nn.Module):
    def __init__(self, n_point, n_attr, loc_dim=32, hidden=128, n_layers=2, dropout=0.2):
        super().__init__()
        self.loc_proj   = nn.Sequential(nn.Linear(n_point, loc_dim), nn.Tanh())
        self.lstm       = nn.LSTM(loc_dim+n_attr, hidden, n_layers, batch_first=True,
                                  dropout=dropout if n_layers>1 else 0.)
        self.local_head = nn.Sequential(
            nn.Linear(hidden,64), nn.LeakyReLU(),
            nn.Linear(64,32),    nn.LeakyReLU(),
            nn.Linear(32,1))
        self.attn_q      = nn.Linear(n_attr, hidden)
        self.entire_head = nn.Sequential(
            nn.Linear(hidden+n_attr, hidden), nn.LeakyReLU(),
            nn.Linear(hidden, hidden//2),     nn.LeakyReLU(),
            nn.Linear(hidden//2, 1))
    def forward(self, X_seq, attr, mask):
        B, T, _ = X_seq.shape
        h, _ = self.lstm(torch.cat([self.loc_proj(X_seq),
                                     attr.unsqueeze(1).expand(-1,T,-1)], -1))
        y_loc = F.softplus(self.local_head(h).squeeze(-1))
        q   = self.attn_q(attr).unsqueeze(-1)
        w   = torch.softmax(torch.bmm(h,q).squeeze(-1).masked_fill(mask==0,-1e9), 1)
        ctx = (h * w.unsqueeze(-1)).sum(1)
        return F.softplus(self.entire_head(torch.cat([ctx,attr],-1)).squeeze(-1)), y_loc, w

def _campbell(s):
    s = np.asarray(s, float)
    a,b,c,d,e = -1.4579,22.0787,76.3271,0.0525,3.2002e-4
    return np.maximum(c/(np.pi*b*(1+((s+a)/b)**2))+d+e*s, 0.01)*3.6

def _ability(front_df):
    v = front_df[front_df['speed_kmh'].between(0.3, 12.)].copy()
    mult = 1.0
    if len(v) >= 5:
        mult = float(np.median(np.clip(
            v['speed_kmh'].values / np.maximum(_campbell(v['signed_slope_deg'].values), 0.1),
            0.2, 4.0)))
    vh = _campbell(np.array([-30,-20,-10,0,10,20,30], float)) * mult
    return dict(zip(['vhat_n30','vhat_n20','vhat_n10','vhat_0','vhat_p10','vhat_p20','vhat_p30'], vh))

ckpt_lstm  = torch.load(str(MODELS_DIR / 'hiking_tte_gis.pt'), map_location='cpu', weights_only=False)
lstm_model = HikingTTEGIS(**ckpt_lstm['model_config'])
lstm_model.load_state_dict(ckpt_lstm['model_state_dict'])
lstm_model.eval()
# явно float32 - x_mean из чекпоинта float64, numpy апкастит при вычитании
x_mean = ckpt_lstm['x_scaler_mean'].astype(np.float32)
x_std  = ckpt_lstm['x_scaler_std'].astype(np.float32)
a_mean = ckpt_lstm['a_scaler_mean'].astype(np.float32)
a_std  = ckpt_lstm['a_scaler_std'].astype(np.float32)
lstm_tte_m = ckpt_lstm['test_metrics']

if 'seg_idx' not in df.columns:
    df['seg_idx'] = df.groupby('track_id').cumcount()

lstm_speed = np.full(len(df), np.nan)

@torch.no_grad()
def _infer_back(back_df, attr_vec):
    X = ((back_df[LSTM_POINT_FEAT].fillna(0).values.astype(np.float32) - x_mean)
         / np.maximum(x_std, 1e-8))
    A = (attr_vec - a_mean) / np.maximum(a_std, 1e-8)
    _, y_loc, _ = lstm_model(torch.tensor(X[None]),
                              torch.tensor(A[None]),
                              torch.ones(1, len(X)))
    t_h = y_loc[0].numpy()
    return back_df['dist_m'].values / (np.maximum(t_h, 1e-4) * 1000)

t0 = _time.time()
print('LSTM инференс...')
for tid, gdf in df[df['track_id'].isin(te_ids)].groupby('track_id'):
    gdf = gdf.sort_values('seg_idx').reset_index(drop=False)
    n   = len(gdf)
    if n < _MIN_S: continue
    sp   = max(3, int(n * _SPLIT))
    back = gdf.iloc[sp:]
    if len(back) < 5: continue
    ele   = gdf['elev_diff_m'].cumsum().values
    attrs = dict(elemax=float(ele.max()), elemin=float(ele.min()),
                 D_plus=float(np.maximum(gdf['elev_diff_m'].values,0).sum()),
                 D_minus=float(np.maximum(-gdf['elev_diff_m'].values,0).sum()),
                 disttotal=float(gdf['dist_m'].sum()))
    attrs.update(_ability(gdf.iloc[:sp]))
    attr_vec = np.array([attrs[k] for k in LSTM_ATTR_FEAT], np.float32)
    lstm_speed[back['index'].values] = _infer_back(back, attr_vec)

print(f'Готово за {_time.time()-t0:.1f}с  ({(~np.isnan(lstm_speed)).sum():,} сегм.)')

yp_lstm_all = lstm_speed[te_idx]
lstm_mask   = ~np.isnan(yp_lstm_all)
yp_lstm     = yp_lstm_all[lstm_mask]
yt_lstm     = y_te.values[lstm_mask]
r2_lstm  = r2_score(yt_lstm, yp_lstm)
mae_lstm = mean_absolute_error(yt_lstm, yp_lstm)
print(f'LSTM speed: MAE={mae_lstm:.3f}  R^2={r2_lstm:.3f}  ({lstm_mask.sum():,}/{len(y_te):,} сегм.)')
print(f'LSTM TTE:   MAPE={lstm_tte_m["MAPE_entire"]:.2f}%  MAE={lstm_tte_m["MAE_entire_h"]:.4f}ч')

## 2. Сравнительная таблица

In [ ]:
# === TTE MAPE для всех моделей (единый таргет: время всего маршрута) ===

_SPLIT    = 0.3
_MIN_SEGS = 20
_SPD_MIN  = 0.3

# test track IDs (GroupShuffleSplit разбивает по трекам целиком)
te_track_ids = set(df.iloc[te_idx]['track_id'].unique())

def tte_mape_fn(predict_speed_fn):
    p_list, a_list = [], []
    for tid, gdf in df[df['track_id'].isin(te_track_ids)].groupby('track_id'):
        gdf = gdf.sort_values('seg_idx').reset_index(drop=True)
        n   = len(gdf)
        if n < _MIN_SEGS: continue
        sp   = max(3, int(n * _SPLIT))
        back = gdf.iloc[sp:]
        if len(back) < 5: continue
        # actual TTE: time_h = dist_km / speed_kmh
        a_h = (back['dist_m'].values / 1000 / np.maximum(back['speed_kmh'].values, _SPD_MIN)).sum()
        # predicted TTE
        p_spd = np.maximum(predict_speed_fn(back), _SPD_MIN)
        p_h   = (back['dist_m'].values / 1000 / p_spd).sum()
        p_list.append(p_h)
        a_list.append(a_h)
    p, a = np.array(p_list), np.array(a_list)
    mae_h = float(np.mean(np.abs(p - a)))
    mape_ = 100 * float(np.mean(np.abs((p - a) / np.maximum(a, 1e-3))))
    return mape_, mae_h

print('Считаем TTE MAPE...')
mape_tob,  mae_tob  = tte_mape_fn(lambda d: tobler_kmh(d['signed_slope_deg'].values))
mape_base, mae_base = tte_mape_fn(lambda d: model_base.predict(d[FEATURES]))
mape_opt,  mae_opt  = tte_mape_fn(lambda d: model_opt.predict(d[FEATURES]))

# LSTM - из чекпоинта (обучался напрямую минимизировать TTE)
mape_lstm = lstm_tte_m['MAPE_entire']
mae_lstm_tte = lstm_tte_m['MAE_entire_h']

# HikingTTE paper (Asako et al., 2021) - внешний результат
PAPER_MAPE = 15.62

print()
print(f'=== Сравнение моделей: TTE MAPE (единый таргет) ===')
print(f'{"Модель":<35} {"TTE MAPE":>9}  {"MAE TTE (ч)":>12}  {"Таргет обучения"}')
print('-' * 82)
rows_tte = [
    ('Tobler (базовый)',           mape_tob,   mae_tob,      'формула'),
    ('HikingTTE (Asako et al.)',   PAPER_MAPE, float('nan'), 'TTE (paper)'),
    ('CatBoost base (GIS)',        mape_base,  mae_base,     'speed_kmh (сегмент)'),
    ('CatBoost opt (GIS + Optuna)',mape_opt,   mae_opt,      'speed_kmh (сегмент)'),
    ('LSTM + GIS (NB-06)',         mape_lstm,  mae_lstm_tte, 'TTE (весь маршрут)'),
]
for name, mp, mh, tgt in rows_tte:
    mh_str = f'{mh:.4f}' if not (mh != mh) else '  -   '
    print(f'{name:<35} {mp:>8.2f}%  {mh_str:>12}  {tgt}')

print()
print(f'Улучшение LSTM vs Tobler:       {mape_tob - mape_lstm:+.1f}пп')
print(f'Улучшение LSTM vs CatBoost-opt: {mape_opt - mape_lstm:+.1f}пп')
print()
print('--- Дополнительно: метрики скорости на сегментах (информативно) ---')
def seg_mape(yt, yp):
    m = yt > 0.1
    return 100 * np.mean(np.abs((yp[m] - yt[m]) / yt[m]))
yt = y_te.values
print(f'{"Модель":<35} {"MAE км/ч":>9}  {"R^2":>6}  {"MAPE скор.":>10}')
print('-' * 66)
for name, yp_s in [
    ('Tobler',                    tobler_kmh(X_te['signed_slope_deg'].values)),
    ('CatBoost base (GIS)',       yp_base),
    ('CatBoost opt (GIS+Optuna)', yp_opt),
]:
    print(f'{name:<35} {mean_absolute_error(yt,yp_s):>9.3f}  {r2_score(yt,yp_s):>6.3f}  {seg_mape(yt,yp_s):>9.1f}%')
yp_l = yp_lstm_all[lstm_mask]
yt_l = y_te.values[lstm_mask]
print(f'{"LSTM + GIS (производная speed)":<35} {mean_absolute_error(yt_l,yp_l):>9.3f}  {r2_score(yt_l,yp_l):>6.3f}  {seg_mape(yt_l,yp_l):>9.1f}%  <- не таргет LSTM')


## 3. Feature Importance

In [ ]:
fi = pd.Series(
    model_opt.get_feature_importance(),
    index=FEATURES
).sort_values(ascending=True)

colors = ['#ef4444' if f in CAT_F else '#3b82f6' for f in fi.index]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(fi.index, fi.values, color=colors)
for bar, val in zip(bars, fi.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)
ax.set_xlabel('Feature Importance (%)')
ax.set_title('CatBoost-opt: Feature Importance')
# Легенда
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#3b82f6', label='Непрерывные'),
    Patch(color='#ef4444', label='Категориальные (GIS)'),
], loc='lower right')
plt.tight_layout()
plt.savefig('figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Кривые скорость–уклон: модель vs Tobler vs Campbell

In [ ]:
slopes = np.linspace(-40, 40, 200)

def make_grid(slopes, on_trail=0, on_road=0, near_water=0):
    off = 1 - max(on_trail, on_road)
    sl  = np.abs(slopes)
    road, trail = bool(on_road), bool(on_trail)
    base = np.where(road, 1,
           np.where(trail & (sl < 15), 2,
           np.where(trail | (sl < 25), 3, 4)))
    cd = np.where(near_water & ~road, np.minimum(base + 1, 5), base)
    dt = np.log1p(0 if on_trail else 200)
    dr = np.log1p(0 if on_road  else 2000)
    rows = []
    for s, b, c in zip(slopes, base, cd):
        rows.append({
            'signed_slope_deg': s, 'elev_diff_m': s * 10,
            'on_trail': on_trail, 'on_road': on_road,
            'off_trail': off, 'near_water': near_water,
            'log_dist_trail': dt, 'log_dist_road': dr,
            'custom_difficulty': int(c),
        })
    return pd.DataFrame(rows)

scenarios = [
    ('Бездорожье',   make_grid(slopes, 0, 0, 0), '#6b7280'),
    ('Тропа (OSM)',  make_grid(slopes, 1, 0, 0), '#a16207'),
    ('Дорога (OSM)', make_grid(slopes, 0, 1, 0), '#1d4ed8'),
    ('Рядом вода',   make_grid(slopes, 0, 0, 1), '#0ea5e9'),
]

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(slopes, tobler_kmh(slopes), 'k--', lw=1.5, label='Tobler', alpha=0.7)

for label, grid, color in scenarios:
    pred = model_opt.predict(grid)
    ax.plot(slopes, pred, color=color, lw=2, label=f'CB: {label}')

# LSTM: медиана предсказаний по бинам уклона (реальные данные, не синтетика)
df_tmp = df.copy()
df_tmp['_lstm'] = lstm_speed
df_valid = df_tmp[~np.isnan(df_tmp['_lstm'])].copy()
bins = np.arange(-40, 41, 2)
df_valid['_bin'] = pd.cut(df_valid['signed_slope_deg'], bins=bins,
                           labels=(bins[:-1] + bins[1:]) / 2)
lstm_curve = df_valid.groupby('_bin', observed=True)['_lstm'].median()
ax.plot(lstm_curve.index.astype(float), lstm_curve.values,
        color='#7c3aed', lw=2.5, linestyle='-.', label='LSTM + GIS (медиана по данным)', alpha=0.9)

ax.set_xlabel('Уклон (deg)  [отрицательный = спуск]')
ax.set_ylabel('Скорость (км/ч)')
ax.set_title('Кривые скорость-уклон: CatBoost vs LSTM vs классические модели')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(-40, 40); ax.set_ylim(0, 10)
ax.axvline(0, color='gray', lw=0.5); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/speed_slope_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Таблица при ключевых уклонах
key_slopes = [-20, -10, 0, 10, 20]
print(f'\n{"Уклон":>8} | {"Tobler":>7} | {"CB-off":>6} | {"CB-тропа":>8} | {"LSTM медиана":>12}')
print('-' * 55)
for s in key_slopes:
    p_off  = float(model_opt.predict(make_grid([s], 0, 0, 0))[0])
    p_tr   = float(model_opt.predict(make_grid([s], 1, 0, 0))[0])
    # LSTM медиана в ближайшем бине
    b_mask = (df_valid['signed_slope_deg'].between(s-2, s+2)) & (~np.isnan(df_valid['_lstm']))
    lstm_med = df_valid.loc[b_mask, '_lstm'].median() if b_mask.sum() > 0 else float('nan')
    print(f'{s:>7}deg | {tobler_kmh(s):>7.2f} | {p_off:>6.2f} | {p_tr:>8.2f} | {lstm_med:>12.2f}')

## 5. Partial Dependence Plots

In [ ]:
from catboost import Pool

# Берём подвыборку test для PDP
sample_idx = np.random.default_rng(42).choice(len(X_te), min(5000, len(X_te)), replace=False)
X_sample = X_te.iloc[sample_idx].reset_index(drop=True)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# PDP для signed_slope_deg
slope_vals = np.linspace(-35, 35, 60)
pdp_slope  = []
for sv in slope_vals:
    Xt = X_sample.copy()
    Xt['signed_slope_deg'] = sv
    pdp_slope.append(model_opt.predict(Xt).mean())
axes[0, 0].plot(slope_vals, pdp_slope, '#3b82f6', lw=2)
axes[0, 0].plot(slope_vals, tobler_kmh(slope_vals), 'k--', lw=1, label='Tobler')
axes[0, 0].set_title('PDP: уклон')
axes[0, 0].set_xlabel('Уклон (deg)')
axes[0, 0].set_ylabel('Средняя скорость (км/ч)')
axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

# PDP для log_dist_trail
dist_vals = np.linspace(0, np.log1p(2000), 50)
pdp_trail = []
for dv in dist_vals:
    Xt = X_sample.copy()
    Xt['log_dist_trail'] = dv
    Xt['on_trail'] = (np.expm1(dv) < 50).astype(int)
    Xt['off_trail'] = ((Xt['on_trail'] == 0) & (Xt['on_road'] == 0)).astype(int)
    pdp_trail.append(model_opt.predict(Xt).mean())
axes[0, 1].plot(np.expm1(dist_vals), pdp_trail, '#a16207', lw=2)
axes[0, 1].set_title('PDP: расстояние до ближайшей тропы')
axes[0, 1].set_xlabel('Дистанция до тропы (м)')
axes[0, 1].set_ylabel('Средняя скорость (км/ч)')
axes[0, 1].grid(alpha=0.3)

# PDP для log_dist_road
dist_r_vals = np.linspace(0, np.log1p(5000), 50)
pdp_road = []
for dv in dist_r_vals:
    Xt = X_sample.copy()
    Xt['log_dist_road'] = dv
    Xt['on_road'] = (np.expm1(dv) < 30).astype(int)
    Xt['off_trail'] = ((Xt['on_trail'] == 0) & (Xt['on_road'] == 0)).astype(int)
    pdp_road.append(model_opt.predict(Xt).mean())
axes[0, 2].plot(np.expm1(dist_r_vals), pdp_road, '#1d4ed8', lw=2)
axes[0, 2].set_title('PDP: расстояние до дороги')
axes[0, 2].set_xlabel('Дистанция до дороги (м)')
axes[0, 2].set_ylabel('Средняя скорость (км/ч)')
axes[0, 2].grid(alpha=0.3)

# Скорость по типу поверхности (box plot)
terrain_labels = {1: 'Дорога', 2: 'Тропа\n(пологая)', 3: 'Тропа\n(крутая)/\ лес', 4: 'Бездорожье', 5: 'Труднодоступно'}
groups_cd = {}
for cd_val in [1, 2, 3, 4, 5]:
    mask = (df.iloc[te_idx]['custom_difficulty'] == cd_val).values
    if mask.sum() > 10:
        groups_cd[terrain_labels.get(cd_val, str(cd_val))] = y_te.values[mask]
axes[1, 0].boxplot(groups_cd.values(), labels=groups_cd.keys(), patch_artist=True,
                    medianprops={'color': 'red', 'linewidth': 2})
axes[1, 0].set_title('Распределение скорости по типу местности')
axes[1, 0].set_ylabel('Скорость (км/ч)')
axes[1, 0].set_ylim(0, 12)
axes[1, 0].grid(alpha=0.3, axis='y')

# Predicted vs Actual
rng_idx = np.random.default_rng(0).choice(len(y_te), min(4000, len(y_te)), replace=False)
axes[1, 1].scatter(y_te.values[rng_idx], yp_opt[rng_idx], s=2, alpha=0.3, c='#ef4444', rasterized=True)
lim = max(y_te.max(), yp_opt.max())
axes[1, 1].plot([0, lim], [0, lim], 'k--', lw=1)
axes[1, 1].set_xlabel('Факт (км/ч)')
axes[1, 1].set_ylabel('Предсказание (км/ч)')
axes[1, 1].set_title(f'Predicted vs Actual  (R^2={r2_score(y_te,yp_opt):.3f})')
axes[1, 1].grid(alpha=0.3)

# Ошибки по on_trail / off_trail / on_road
te_df = df.iloc[te_idx].copy()
te_df['error'] = np.abs(y_te.values - yp_opt)
cat_means = {
    'off_trail': te_df[te_df['off_trail'] == 1]['error'].mean(),
    'on_trail':  te_df[te_df['on_trail'] == 1]['error'].mean(),
    'on_road':   te_df[te_df['on_road'] == 1]['error'].mean(),
    'near_water': te_df[te_df['near_water'] == 1]['error'].mean(),
}
axes[1, 2].bar(cat_means.keys(), cat_means.values(), color=['#6b7280','#a16207','#1d4ed8','#0ea5e9'])
axes[1, 2].set_title('MAE по типу местности')
axes[1, 2].set_ylabel('MAE (км/ч)')
axes[1, 2].grid(alpha=0.3, axis='y')
for k, v in zip(cat_means.keys(), cat_means.values()):
    axes[1, 2].text(list(cat_means.keys()).index(k), v + 0.01, f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('figures/catboost_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. SHAP-анализ

In [ ]:
!pip install shap

In [ ]:
try:
    import shap
    HAS_SHAP = True
except ImportError:
    print('pip install shap')
    HAS_SHAP = False

if HAS_SHAP:
    sample_shap = X_te.iloc[:2000].reset_index(drop=True)

    explainer = shap.TreeExplainer(model_opt)
    shap_vals = explainer.shap_values(sample_shap)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Summary plot (beeswarm)
    plt.sca(axes[0])
    shap.summary_plot(shap_vals, sample_shap, feature_names=FEATURES,
                      show=False, plot_size=None)
    axes[0].set_title('SHAP: влияние признаков на скорость')

    # Bar plot: mean |SHAP|
    mean_shap = np.abs(shap_vals).mean(axis=0)
    order = np.argsort(mean_shap)
    axes[1].barh([FEATURES[i] for i in order], mean_shap[order], color='#3b82f6')
    axes[1].set_title('SHAP: mean |SHAP value|')
    axes[1].set_xlabel('mean |SHAP| (км/ч)')

    plt.tight_layout()
    plt.savefig('figures/shap_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

    print('\nSHAP mean |value| по признакам:')
    for f, v in sorted(zip(FEATURES, mean_shap), key=lambda x: -x[1]):
        print(f'  {f:<25} {v:.4f}')
else:
    print('SHAP не установлен. Запустите: !pip install shap')

## 7. Эффект тропы: +X% к скорости vs бездорожья

In [ ]:
# Для каждого сегмента off_trail - предсказываем скорость в двух режимах
te_df_off = X_te[X_te['off_trail'] == 1].copy()
print(f'Off-trail сегментов: {len(te_df_off):,}')

# Предсказание как есть (off_trail)
v_off = model_opt.predict(te_df_off)

# Предсказание если бы там была тропа
te_on_trail = te_df_off.copy()
te_on_trail['on_trail']       = 1
te_on_trail['off_trail']      = 0
te_on_trail['log_dist_trail'] = np.log1p(5)  # 5м до тропы
# обновляем custom_difficulty: было 4+, будет 2-3
sl = te_on_trail['signed_slope_deg'].abs().values
te_on_trail['custom_difficulty'] = np.where(sl < 15, 2, 3)
v_on_trail = model_opt.predict(te_on_trail)

# Предсказание если бы там была дорога
te_on_road = te_df_off.copy()
te_on_road['on_road']        = 1
te_on_road['off_trail']      = 0
te_on_road['log_dist_road']  = np.log1p(5)
te_on_road['custom_difficulty'] = 1
v_on_road = model_opt.predict(te_on_road)

trail_boost = (v_on_trail - v_off) / v_off * 100
road_boost  = (v_on_road  - v_off) / v_off * 100

print(f'\nЭффект тропы vs бездорожья:')
print(f'  Медиана +{np.median(trail_boost):.1f}%  (p25={np.percentile(trail_boost,25):.1f}%, p75={np.percentile(trail_boost,75):.1f}%)')
print(f'Эффект дороги vs бездорожья:')
print(f'  Медиана +{np.median(road_boost):.1f}%  (p25={np.percentile(road_boost,25):.1f}%, p75={np.percentile(road_boost,75):.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, boost, label, color in [
    (axes[0], trail_boost, 'Тропа vs бездорожье', '#a16207'),
    (axes[1], road_boost,  'Дорога vs бездорожье', '#1d4ed8'),
]:
    ax.hist(np.clip(boost, -50, 150), bins=60, color=color, alpha=0.8, edgecolor='white', lw=0.3)
    ax.axvline(np.median(boost), color='red', lw=2, label=f'Медиана: +{np.median(boost):.1f}%')
    ax.axvline(0, color='black', lw=1, ls='--')
    ax.set_title(f'Прирост скорости: {label}')
    ax.set_xlabel('Прирост скорости (%)')
    ax.legend()
plt.tight_layout()
plt.savefig('figures/trail_road_effect.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Итоговые выводы для диссертации

In [ ]:
print('=' * 65)
print('ИТОГ: сравнение моделей (TTE MAPE - основной таргет)')
print('=' * 65)
print()
print(f'Tobler (базовый):             TTE MAPE = {mape_tob:.2f}%  MAE = {mae_tob:.4f} ч')
print(f'HikingTTE (Asako et al.):     TTE MAPE = {PAPER_MAPE:.2f}%  (внешний)')
print(f'CatBoost base (GIS):          TTE MAPE = {mape_base:.2f}%  MAE = {mae_base:.4f} ч')
print(f'CatBoost opt (GIS + Optuna):  TTE MAPE = {mape_opt:.2f}%  MAE = {mae_opt:.4f} ч')
print(f'LSTM + GIS (NB-06):           TTE MAPE = {mape_lstm:.2f}%  MAE = {mae_lstm_tte:.4f} ч  <- обучался на TTE')
print()
print(f'CatBoost vs Tobler:  {mape_tob - mape_opt:+.1f}пп TTE MAPE,  MAE скорости -{mean_absolute_error(y_te.values, tobler_kmh(X_te["signed_slope_deg"].values)) - mean_absolute_error(y_te.values, yp_opt):.3f} км/ч')
print(f'LSTM vs CatBoost-opt:{mape_opt - mape_lstm:+.1f}пп TTE MAPE  (TTE - прямой таргет обучения LSTM)')
print()
print(f'Эффект тропы vs бездорожья:  медиана +{np.median(trail_boost):.0f}% (CatBoost)')
print(f'Эффект дороги vs бездорожья: медиана +{np.median(road_boost):.0f}% (CatBoost)')
print()
print('Файлы:')
print('  models/travel_time_opt.cbm  - CatBoost (speed_kmh per segment)')
print('  models/hiking_tte_gis.pt    - LSTM (TTE + per-segment time)')
